# Optimización y Simulación de Despachos

- **Autor:** Luis Miguel Marín Cadavid
- **Fecha:** 16 Julio 2026
- **Fase 3:** Se convierten los resultados predictivos de la Fase 2 en decisiones automatizadas y óptimas en los despachos

## Estructura del Notebook

1. Configuración del entorno
2. Carga de datos y modelos
3. Modelo de Optimización con PuLP
4. Simulación de Eventos Discretos con SimPy
5. Pipeline principal
6. Ejecución
7. Análisis y Exportación de Resultados
8. Conclusiones

## 1. Configuración del entorno

Se importan todas las librerías necesarias para la Fase 3

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import sys
import subprocess
import warnings
from datetime import datetime

# sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

# Estilo de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Entorno configurado")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy:  {np.__version__}")

In [ ]:

# Instalación de dependencias específicas
def instalar_paquete(paquete):
    """Instala un paquete si no está disponible"""
    try:
        __import__(paquete)
        print(f"{paquete} ya está instalado")
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
        print(f"{paquete} instalado")

instalar_paquete('pulp')
instalar_paquete('simpy')

import pulp
import simpy

print("\nTodas las dependencias listas")


## 2. Carga de Datos y Modelos

Centralizar la información suministrada en las fases anteriores:
1. **Datos procesados**: Dataset limpio con features de ingeniería
2. **Modelo Random Forest**: Predictor de tiempos de preparación
3. **Scaler**: Transformación de features
4. **Metadatos**: Información del modelo


In [ ]:
class DataLoaderExacto:
    """
    Carga los datos procesados y modelos entrenados en la Fase 2.
    Reconstruye las features exactamente como el scaler las espera.
    """
    
    def __init__(self):
        self.data_path = '../data/processed/despachos_clean.csv'
        self.model_path = '../models/random_forest_model.pkl'
        self.scaler_path = '../models/scaler.pkl'
        self.metadata_path = '../models/metadatos.pkl'
        self.model = None
        self.scaler = None
        self.metadata = None
        self.df_completo = None
    
    def load_data(self):
        """Carga todos los artefactos de fases anteriores"""
        print("Cargando datos y modelos...")
        print("-" * 40)
        
        # Datos
        self.df_completo = pd.read_csv(self.data_path)
        print(f"Dataset: {len(self.df_completo)} registros")
        print(f"Fechas únicas: {self.df_completo['fecha'].nunique()} días")
        print(f"Rango: {self.df_completo['fecha'].min()} → {self.df_completo['fecha'].max()}")
        
        # Modelo
        with open(self.model_path, 'rb') as f:
            self.model = pickle.load(f)
        print(f"Modelo: {type(self.model).__name__}")
        
        # Scaler
        with open(self.scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)
        print(f"Scaler: {type(self.scaler).__name__}")
        print(f"Features esperadas: {self.scaler.n_features_in_}")
        
        # Metadatos
        with open(self.metadata_path, 'rb') as f:
            self.metadata = pickle.load(f)
        print(f"Metadatos cargados")
        
        return self.df_completo, self.model, self.scaler, self.metadata
    
    def get_features(self, df):
        """
        Reconstruye las features exactamente como el scaler las espera.
        Maneja one-hot encoding de días de semana automáticamente.
        """
        df_proc = df.copy()
        
        # One-hot encoding para días de semana
        if 'dia_semana' in df_proc.columns:
            dias_semana = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
            for dia in dias_semana:
                col = f'dia_semana_{dia}'
                df_proc[col] = (df_proc['dia_semana'] == dia).astype(int)
        
        # Asegurar que todas las features esperadas existen
        features_esperadas = list(self.scaler.feature_names_in_)
        for feat in features_esperadas:
            if feat not in df_proc.columns:
                df_proc[feat] = 0
        
        return df_proc[features_esperadas]
    
    def predecir_tiempos(self, df):
        """
        Predice el tiempo de preparación para cada ruta.
        Los tiempos representan cuánto tardaría LUIS (operario 100% eficiente).
        """
        features = self.get_features(df)
        X_scaled = self.scaler.transform(features)
        tiempos = self.model.predict(X_scaled)
        tiempos = np.maximum(tiempos, 1)  # Mínimo 1 minuto
        return tiempos
    
    def obtener_rutas_dia(self, fecha=None):
        """
        Obtiene las rutas de un día específico.
        Si no se especifica fecha, toma un día aleatorio.
        """
        if fecha is None:
            fecha = np.random.choice(self.df_completo['fecha'].unique())
        
        df_dia = self.df_completo[self.df_completo['fecha'] == fecha].reset_index(drop=True)
        
        if len(df_dia) == 0:
            raise ValueError(f"No hay datos para la fecha {fecha}")
        
        return fecha, df_dia
    
    def obtener_info_dia(self, fecha):
        """Muestra información resumida de un día"""
        df_dia = self.df_completo[self.df_completo['fecha'] == fecha]
        
        info = {
            'fecha': fecha,
            'n_rutas': len(df_dia),
            'productos_totales': df_dia['cant_productos'].sum() if 'cant_productos' in df_dia.columns else 0,
            'valor_total': df_dia['valor_ruta'].sum() if 'valor_ruta' in df_dia.columns else 0,
            'dia_semana': df_dia['dia_semana'].iloc[0] if 'dia_semana' in df_dia.columns else 'N/A'
        }
        return info


# Verificación rápida
print("=" * 60)
print("VERIFICANDO DATA LOADER")
print("=" * 60)

loader = DataLoaderExacto()
df, model, scaler, metadata = loader.load_data()

# Probar predicción
fecha_test, rutas_test = loader.obtener_rutas_dia()
tiempos_test = loader.predecir_tiempos(rutas_test)
print(f"\nPredicción de prueba: {len(tiempos_test)} rutas")
print(f"   Tiempo promedio: {np.mean(tiempos_test):.1f} min")
print(f"   Tiempo total: {np.sum(tiempos_test):.0f} min ({np.sum(tiempos_test)/60:.1f} horas)")

## 3. Modelo de Optimización con PuLP


### Variables de Decisión
- $x_{ij} \in \{0,1\}$: 1 si la ruta $i$ se asigna al operario $j$

### Función Objetivo
$$\min C_{max}$$

### Restricciones
1. **Asignación única**: $\sum_{j} x_{ij} = 1 \quad \forall i$
2. **Balance de carga**: $\sum_{i} (t_i / e_j) \cdot x_{ij} \leq C_{max} \quad \forall j$
3. **Rutas estrella**: $x_{i,0} = 1 \quad \forall i \in \text{Estrella}$

### Parámetros
- $t_i$: Tiempo predicho para ruta $i$
- $e_j$: Eficiencia del operario $j$ (1.0 = óptimo)

In [ ]:
class AsignadorDiario:
    """
    Sistema que recibe las rutas de una jornada y las reparte
    entre los operarios de forma balanceada.
    
    Principios:
    - Cada operario tiene una eficiencia relativa a Luis
    - Las rutas se asignan para igualar la hora de salida
    - Rutas más largas se asignan primero (evita desbalances)
    
    Operarios:
    - Luis (TÚ):      100% - Referencia, el más rápido
    - Operario 2:      85% - 15% más lento que Luis
    - Operario 3:      75% - 25% más lento que Luis
    """
    
    def __init__(self, loader):
        """
        Args:
            loader: Instancia de DataLoaderExacto con modelos cargados
        """
        self.loader = loader
        
        # Definición de operarios y su eficiencia
        self.operarios = {
            'Luis (TÚ)':   1.00,
            'Operario 2':  0.85,
            'Operario 3':  0.75
        }
        
        self.nombres = list(self.operarios.keys())
        self.eficiencias = list(self.operarios.values())
        self.n_operarios = len(self.operarios)
    
    def asignar_jornada(self, rutas_hoy, usar_pulp=False):
        """
        Asigna las rutas de una jornada a los operarios.
        
        Args:
            rutas_hoy: DataFrame con las rutas del día
            usar_pulp: Si True, usa optimización exacta (más lenta)
                      Si False, usa algoritmo greedy (instantáneo)
        
        Returns:
            asignacion: dict con lista de rutas por operario
            cargas: array con carga en minutos por operario
            makespan: tiempo del operario que más tarda
            tiempos_base: tiempos predichos para cada ruta
        """
        n_rutas = len(rutas_hoy)
        
        print(f"\nAnalizando {n_rutas} rutas...")
        
        # PASO 1: Predecir tiempos base (a velocidad de Luis)
        tiempos_base = self.loader.predecir_tiempos(rutas_hoy)
        tiempo_total = np.sum(tiempos_base)
        
        print(f"Tiempo total de trabajo: {tiempo_total:.0f} min ({tiempo_total/60:.1f} h)")
        print(f"Promedio por ruta: {np.mean(tiempos_base):.1f} min")
        
        # PASO 2: Calcular matriz de tiempos efectivos
        tiempos_ef = np.zeros((n_rutas, self.n_operarios))
        for i in range(n_rutas):
            for j in range(self.n_operarios):
                tiempos_ef[i, j] = tiempos_base[i] / self.eficiencias[j]
        
        # PASO 3: Asignar rutas
        if usar_pulp and n_rutas <= 100:
            asignacion, cargas = self._optimizar_pulp(tiempos_ef, tiempos_base)
        else:
            asignacion, cargas = self._optimizar_greedy(tiempos_ef, tiempos_base)
        
        # PASO 4: Construir resultado detallado
        resultado = {nombre: [] for nombre in self.nombres}
        
        for j, nombre in enumerate(self.nombres):
            for idx in asignacion[j]:
                resultado[nombre].append({
                    'ruta_id': rutas_hoy.iloc[idx].get('id_ruta', f'R{idx}'),
                    'tiempo_base': round(tiempos_base[idx], 1),
                    'tiempo_asignado': round(tiempos_ef[idx, j], 1),
                    'productos': rutas_hoy.iloc[idx].get('cant_productos', '-'),
                    'valor': rutas_hoy.iloc[idx].get('valor_ruta', '-')
                })
        
        makespan = max(cargas)
        
        return resultado, cargas, makespan, tiempos_base
    
    def _optimizar_greedy(self, tiempos_ef, tiempos_base):
        """
        Algoritmo Greedy balanceado:
        1. Ordena rutas de mayor a menor tiempo
        2. Asigna cada ruta al operario con menor carga actual
        
        Ventajas: Instantáneo, buenos resultados
        """
        n_rutas = len(tiempos_base)
        orden = np.argsort(tiempos_base)[::-1]  # Más largas primero
        
        cargas = np.zeros(self.n_operarios)
        asignacion = [[] for _ in range(self.n_operarios)]
        
        for idx in orden:
            op = np.argmin(cargas)  # Operario más libre
            cargas[op] += tiempos_ef[idx, op]
            asignacion[op].append(idx)
        
        print(f"Asignación Greedy completada")
        
        return asignacion, cargas
    
    def _optimizar_pulp(self, tiempos_ef, tiempos_base):
        """
        Optimización exacta con PuLP (Programación Lineal Entera).
        Solo se usa para días con ≤100 rutas por velocidad.
        """
        n_rutas = len(tiempos_base)
        
        # Crear problema
        prob = pulp.LpProblem("Asignacion_Diaria", pulp.LpMinimize)
        
        # Variables de decisión
        x = pulp.LpVariable.dicts("x",
            [(i, j) for i in range(n_rutas) for j in range(self.n_operarios)],
            cat='Binary')
        
        # Variable makespan
        C_max = pulp.LpVariable("C_max", lowBound=0)
        
        # Función objetivo: minimizar makespan
        prob += C_max
        
        # Restricción 1: cada ruta a un operario
        for i in range(n_rutas):
            prob += pulp.lpSum(x[(i, j)] for j in range(self.n_operarios)) == 1
        
        # Restricción 2: carga de cada operario ≤ C_max
        for j in range(self.n_operarios):
            prob += pulp.lpSum(tiempos_ef[i, j] * x[(i, j)] for i in range(n_rutas)) <= C_max
        
        # Resolver con timeout
        solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=30)
        prob.solve(solver)
        
        if pulp.LpStatus[prob.status] in ['Optimal', 'Feasible']:
            cargas = np.zeros(self.n_operarios)
            asignacion = [[] for _ in range(self.n_operarios)]
            
            for i in range(n_rutas):
                for j in range(self.n_operarios):
                    if pulp.value(x[(i, j)]) == 1:
                        cargas[j] += tiempos_ef[i, j]
                        asignacion[j].append(i)
                        break
            
            print(f"Optimización exacta (PuLP) completada")
            return asignacion, cargas
        else:
            # Fallback a Greedy
            print(f"PuLP no convergió, usando Greedy")
            return self._optimizar_greedy(tiempos_ef, tiempos_base)
    
    def mostrar_resultado(self, asignacion, cargas, makespan):
        """Muestra la asignación en formato legible"""
        n_total = sum(len(asignacion[n]) for n in self.nombres)
        
        print(f"\n{'='*65}")
        print(f"ASIGNACIÓN DE LA JORNADA")
        print(f"{'='*65}")
        
        for j, nombre in enumerate(self.nombres):
            n_rutas = len(asignacion[nombre])
            carga = cargas[j]
            pct_rutas = n_rutas / n_total * 100
            emoji = "Estrella" if "Luis" in nombre else "Operador"
            
            print(f"\n{emoji} {nombre}")
            print(f"   {'─'*50}")
            print(f"   Rutas asignadas: {n_rutas} ({pct_rutas:.0f}% del total)")
            print(f"   Carga de trabajo: {carga:.0f} min = {carga/60:.1f} horas")
            
            if makespan > 0:
                print(f"   Ocupación: {carga/makespan*100:.0f}% de la jornada")
            
            # Top 5 rutas más largas
            rutas_ord = sorted(asignacion[nombre], key=lambda x: x['tiempo_asignado'], reverse=True)
            print(f"   Rutas más largas:")
            for r in rutas_ord[:5]:
                print(f"     • Ruta {r['ruta_id']}: {r['tiempo_asignado']:.0f} min "
                      f"(Luis: {r['tiempo_base']:.0f} min)")
            if len(rutas_ord) > 5:
                print(f"     ... y {len(rutas_ord)-5} rutas más")
        
        # Resumen
        print(f"\n{'='*65}")
        print(f"RESUMEN DE LA JORNADA")
        print(f"{'='*65}")
        print(f"   Total rutas: {n_total}")
        print(f"Cierre estimado: {makespan:.0f} min ({makespan/60:.1f} horas)")
        
        if makespan > 480:
            print(f"Horas extra necesarias: {(makespan-480)/60:.1f} h")
        else:
            print(f"Dentro de jornada de 8 horas")
            print(f"Holgura: {(480-makespan)/60:.1f} horas")
        
        cv = np.std(cargas) / np.mean(cargas) * 100
        print(f"Balance de carga: {cv:.1f}% de variación")
        
        # Productividad
        print(f"\nPRODUCTIVIDAD POR OPERARIO:")
        for j, nombre in enumerate(self.nombres):
            rutas_hora = len(asignacion[nombre]) / (cargas[j]/60) if cargas[j] > 0 else 0
            print(f"   {nombre}: {rutas_hora:.1f} rutas/hora")
    
    def visualizar_asignacion(self, asignacion, cargas, makespan, tiempos_base):
        """Genera gráficos de la asignación"""
        colors = ['#2E86AB', '#A23B72', '#F18F01']
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Carga por operario (barras)
        ax1 = axes[0, 0]
        bars = ax1.bar(self.nombres, [c/60 for c in cargas], color=colors, edgecolor='black', linewidth=1.5)
        ax1.axhline(y=8, color='red', linestyle='--', linewidth=2, label='Jornada 8 horas')
        ax1.axhline(y=makespan/60, color='green', linestyle='--', linewidth=2, 
                   label=f'Makespan: {makespan/60:.1f} h')
        ax1.set_ylabel('Horas')
        ax1.set_title('Carga de Trabajo por Operario', fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        
        for bar, carga, nombre in zip(bars, cargas, self.nombres):
            n_rutas = len(asignacion[nombre])
            ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                    f'{n_rutas} rutas\n{carga/60:.1f} h',
                    ha='center', fontweight='bold', fontsize=9)
        
        # 2. Distribución de tiempos por operario
        ax2 = axes[0, 1]
        for j, nombre in enumerate(self.nombres):
            tiempos = [r['tiempo_asignado'] for r in asignacion[nombre]]
            ax2.hist(tiempos, alpha=0.6, bins=15, color=colors[j], label=nombre, edgecolor='black')
        ax2.set_xlabel('Minutos por ruta')
        ax2.set_ylabel('Frecuencia')
        ax2.set_title('Distribución de Tiempos de Ruta', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Tiempos base vs asignados (comparación Luis)
        ax3 = axes[1, 0]
        for j, nombre in enumerate(self.nombres):
            tiempos_asig = [r['tiempo_asignado'] for r in asignacion[nombre]]
            if tiempos_asig:
                ax3.boxplot(tiempos_asig, positions=[j], widths=0.5, 
                           patch_artist=True, boxprops=dict(facecolor=colors[j]))
        ax3.set_xticklabels(self.nombres)
        ax3.set_ylabel('Minutos por ruta')
        ax3.set_title('Comparación de Tiempos por Operario', fontweight='bold')
        ax3.grid(True, alpha=0.3)
        
        # 4. Balance de carga
        ax4 = axes[1, 1]
        cuartiles = np.percentile(tiempos_base, [25, 50, 75])
        etiquetas = ['Rápidas\n(<25%)', 'Medias\n(25-75%)', 'Largas\n(>75%)']
        tamaños = [np.sum(tiempos_base <= cuartiles[0]),
                  np.sum((tiempos_base > cuartiles[0]) & (tiempos_base <= cuartiles[2])),
                  np.sum(tiempos_base > cuartiles[2])]
        ax4.pie(tamaños, labels=etiquetas, autopct='%1.1f%%', 
               colors=['#06A77D', '#F18F01', '#A23B72'], startangle=90)
        ax4.set_title('Composición de Rutas por Duración', fontweight='bold')
        
        plt.suptitle(f'Asignación de Jornada | Makespan: {makespan:.0f} min ({makespan/60:.1f} h)',
                    fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()


## 4. Simulación de Eventos discretos SimPy

### Objetivo
Evaluar la robustez de la asignación bajo diferentes escenarios operativos.

### Escenarios Simulados
1. **Base**: Condiciones normales de operación
2. **Pico**: Incremento del 20% en tiempos de preparación
3. **Ausentismo**: Solo 2 operarios disponibles
4. **Rendimiento Variable**: Eficiencias aleatorias (0.7 - 1.0)

### Métricas de Evaluación
- Makespan (tiempo de cierre)
- Horas extra requeridas
- Equidad en distribución de carga (CV)
- Utilización de recursos
- Impacto del operario más lento

In [ ]:
class SimuladorDespachos:
    """
    Simula la jornada bajo diferentes escenarios para evaluar
    la robustez de la asignación.
    
    Escenarios:
    - base: Condiciones normales
    - pico: +20% en tiempos (alta demanda)
    - ausentismo: Solo 2 operarios (uno falta)
    - rendimiento_variable: Eficiencias aleatorias (día irregular)
    """
    
    def __init__(self, tiempos_rutas, n_operarios=3, eficiencias=[1.0, 0.85, 0.75]):
        self.tiempos = tiempos_rutas
        self.n_operarios = n_operarios
        self.eficiencias = eficiencias
        self.n_rutas = len(tiempos_rutas)
    
    def ejecutar_escenario(self, escenario, seed=None):
        """Simula un escenario individual con SimPy"""
        if seed is not None:
            np.random.seed(seed)
        
        # Configurar según escenario
        if escenario == 'pico':
            t = self.tiempos * 1.2
            n_ops, effs = self.n_operarios, self.eficiencias
        elif escenario == 'ausentismo':
            t = self.tiempos
            n_ops, effs = 2, self.eficiencias[:2]
        elif escenario == 'rendimiento_variable':
            t = self.tiempos
            n_ops = self.n_operarios
            effs = np.random.uniform(0.7, 1.0, self.n_operarios).tolist()
        else:  # base
            t = self.tiempos
            n_ops, effs = self.n_operarios, self.eficiencias
        
        # Simulación SimPy
        env = simpy.Environment()
        cargas = [0.0] * n_ops
        
        def proceso(env):
            for idx in np.random.permutation(self.n_rutas):
                op = min(range(n_ops), key=lambda j: cargas[j])
                tiempo = t[idx] / effs[op]
                yield env.timeout(tiempo)
                cargas[op] += tiempo
        
        env.process(proceso(env))
        env.run()
        
        makespan = max(cargas) if cargas else 0
        
        return {
            'escenario': escenario,
            'n_operarios': n_ops,
            'cargas': cargas,
            'makespan': makespan,
            'horas_extra': max(0, makespan - 480) / 60,
            'cv_carga': np.std(cargas) / np.mean(cargas) if np.mean(cargas) > 0 else 0,
            'utilizacion': [c / makespan if makespan > 0 else 0 for c in cargas]
        }
    
    def simular(self, n_iter=500):
        """Ejecuta Monte Carlo para todos los escenarios"""
        escenarios = ['base', 'pico', 'ausentismo', 'rendimiento_variable']
        resultados = {e: [] for e in escenarios}
        
        print(f"\n🎲 Simulación Monte Carlo ({n_iter} iteraciones/escenario)")
        print("-" * 50)
        
        for esc in escenarios:
            print(f"   {esc.upper():.<25}", end=" ", flush=True)
            for i in range(n_iter):
                resultados[esc].append(self.ejecutar_escenario(esc, seed=i+42))
            print("Bien!")
        
        # Consolidar en DataFrame
        todos = []
        for res in resultados.values():
            todos.extend(res)
        
        return pd.DataFrame(todos)
    
    def visualizar(self, df_resultados):
        """Dashboard de resultados de simulación"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Makespan por escenario
        df_resultados.boxplot(column='makespan', by='escenario', ax=axes[0, 0], patch_artist=True)
        axes[0, 0].axhline(y=480, color='red', linestyle='--', linewidth=2, label='8 horas')
        axes[0, 0].set_title('Makespan por Escenario', fontweight='bold')
        axes[0, 0].set_ylabel('Minutos')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Horas extra
        df_resultados.boxplot(column='horas_extra', by='escenario', ax=axes[0, 1], patch_artist=True)
        axes[0, 1].axhline(y=0, color='green', linestyle='--', linewidth=2, label='Sin extras')
        axes[0, 1].set_title('Horas Extra por Escenario', fontweight='bold')
        axes[0, 1].set_ylabel('Horas')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Balance (CV)
        df_resultados.boxplot(column='cv_carga', by='escenario', ax=axes[1, 0], patch_artist=True)
        axes[1, 0].axhline(y=0.2, color='red', linestyle='--', linewidth=2, label='CV=0.2 (umbral)')
        axes[1, 0].set_title('Balance de Carga (CV)', fontweight='bold')
        axes[1, 0].set_ylabel('Coeficiente de Variación')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Probabilidad de horas extra
        prob_extras = df_resultados.groupby('escenario').apply(
            lambda x: (x['horas_extra'] > 0).sum() / len(x) * 100
        )
        colors = ['#2E86AB', '#A23B72', '#F18F01', '#06A77D']
        axes[1, 1].bar(prob_extras.index, prob_extras.values, color=colors, edgecolor='black')
        axes[1, 1].set_ylabel('% de días')
        axes[1, 1].set_title('Probabilidad de Horas Extra', fontweight='bold')
        axes[1, 1].set_ylim(0, 110)
        
        for i, v in enumerate(prob_extras.values):
            axes[1, 1].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
        
        plt.suptitle('Simulación de Escenarios Operativos', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        
        # Resumen numérico
        print("\nRESUMEN DE SIMULACIÓN:")
        print("=" * 60)
        resumen = df_resultados.groupby('escenario').agg(
            makespan_prom=('makespan', 'mean'),
            makespan_std=('makespan', 'std'),
            horas_extra_prom=('horas_extra', 'mean'),
            prob_horas_extra=('horas_extra', lambda x: (x > 0).sum() / len(x) * 100),
            balance_prom=('cv_carga', 'mean')
        ).round(1)
        print(resumen.to_string())


In [ ]:
class AsignacionJustaOptimizer:
    """
    Optimizador de asignación de rutas usando Programación Lineal Entera (PuLP).
    Versión adaptativa que no requiere columna 'cluster'.
    """
    
    def __init__(self, df_rutas, tiempos_predichos, n_operarios=3, 
                 eficiencias=[1.0, 0.85, 0.75], columna_prioridad=None):
        """
        Args:
            df_rutas: DataFrame con información de rutas
            tiempos_predichos: Array con tiempos predichos por RF
            n_operarios: Número de operarios disponibles
            eficiencias: Factores de eficiencia
            columna_prioridad: Nombre de columna para identificar rutas prioritarias
        """
        self.df_rutas = df_rutas.reset_index(drop=True)
        self.tiempos = tiempos_predichos
        self.n_operarios = n_operarios
        self.eficiencias = eficiencias
        self.n_rutas = len(df_rutas)
        
        # Identificar rutas prioritarias (Estrella)
        self.rutas_estrella = self._identificar_rutas_estrella(columna_prioridad)
        print(f"Rutas Estrella identificadas: {len(self.rutas_estrella)}")
        
        # Operario más eficiente (índice 0)
        self.operario_estrella = 0
        
        # Resultados
        self.resultado = None
        self.asignacion = None
        self.cargas = None
    
    def _identificar_rutas_estrella(self, columna_prioridad=None):
        """
        Identifica rutas prioritarias (Estrella) basado en criterios disponibles.
        
        Estrategias en orden de prioridad:
        1. Usar columna 'cluster' si existe
        2. Usar columna de prioridad especificada
        3. Identificar por valor alto y/o tiempo crítico
        4. Si nada funciona, seleccionar top 20% por valor
        """
        # Estrategia 1: Columna 'cluster'
        if 'cluster' in self.df_rutas.columns:
            return self.df_rutas[self.df_rutas['cluster'] == 1].index.tolist()
        
        # Estrategia 2: Columna de prioridad especificada
        if columna_prioridad and columna_prioridad in self.df_rutas.columns:
            # Asumir que valores altos = prioritario
            umbral = self.df_rutas[columna_prioridad].quantile(0.8)
            return self.df_rutas[self.df_rutas[columna_prioridad] >= umbral].index.tolist()
        
        # Estrategia 3: Identificar por valor monetario (si existe)
        columnas_valor = ['valor_ruta', 'valor', 'monto', 'importe']
        for col in columnas_valor:
            if col in self.df_rutas.columns:
                # Top 20% por valor = rutas estrella
                umbral = self.df_rutas[col].quantile(0.8)
                estrellas = self.df_rutas[self.df_rutas[col] >= umbral].index.tolist()
                print(f"Identificadas por {col} >= {umbral:.0f}")
                return estrellas
        
        # Estrategia 4: Por tiempo predicho (rutas más largas = críticas)
        if self.tiempos is not None and len(self.tiempos) > 0:
            umbral = np.percentile(self.tiempos, 80)
            estrellas = np.where(self.tiempos >= umbral)[0].tolist()
            print(f"Identificadas por tiempo >= {umbral:.1f} min")
            return estrellas
        
        # Estrategia 5: Sin criterio, devolver vacío
        print("No se pudo identificar rutas estrella automáticamente")
        return []
    
    def crear_modelo(self):
        """Crea el modelo de optimización en PuLP"""
        problema = pulp.LpProblem("Asignacion_Justa_Rutas", pulp.LpMinimize)
        
        # Variables de decisión
        x = pulp.LpVariable.dicts("x", 
                                 ((i, j) for i in range(self.n_rutas) 
                                  for j in range(self.n_operarios)),
                                 cat='Binary')
        
        # Variable makespan
        C_max = pulp.LpVariable("C_max", lowBound=0)
        
        # Matriz de tiempos con eficiencia
        tiempos_efectivos = np.zeros((self.n_rutas, self.n_operarios))
        for i in range(self.n_rutas):
            for j in range(self.n_operarios):
                tiempos_efectivos[i, j] = self.tiempos[i] / self.eficiencias[j]
        
        # FUNCIÓN OBJETIVO
        problema += C_max, "Minimizar_makespan"
        
        # RESTRICCIONES
        
        # 1. Cada ruta asignada a un operario
        for i in range(self.n_rutas):
            problema += pulp.lpSum(x[(i, j)] for j in range(self.n_operarios)) == 1
        
        # 2. Carga por operario ≤ C_max
        for j in range(self.n_operarios):
            carga_j = pulp.lpSum(tiempos_efectivos[i, j] * x[(i, j)] 
                               for i in range(self.n_rutas))
            problema += carga_j <= C_max
        
        # 3. Rutas Estrella al operario más eficiente (si hay)
        if self.rutas_estrella:
            for i in self.rutas_estrella:
                problema += x[(i, self.operario_estrella)] == 1
            print(f"{len(self.rutas_estrella)} rutas estrella asignadas al Operario 1")
        else:
            print("Sin restricciones de ruta estrella")
        
        print(f"Modelo: {self.n_rutas} rutas × {self.n_operarios} operarios")
        
        return problema, x, C_max, tiempos_efectivos
    
    def optimizar(self):
        """Ejecuta la optimización"""
        print("\nIniciando optimización...")
        problema, x, C_max, tiempos_efectivos = self.crear_modelo()
        
        # Resolver
        problema.solve(pulp.PULP_CBC_CMD(msg=False))
        
        estado = pulp.LpStatus[problema.status]
        print(f"Estado: {estado}")
        
        if estado == 'Optimal':
            makespan_optimo = pulp.value(C_max)
            
            self.resultado = {
                'status': 'Óptimo',
                'makespan': makespan_optimo,
                'tiempo_ejecucion': problema.solutionTime
            }
            
            # Construir asignación
            asignaciones = []
            for i in range(self.n_rutas):
                for j in range(self.n_operarios):
                    if pulp.value(x[(i, j)]) == 1:
                        # Obtener cluster si existe, sino NaN
                        cluster_val = self.df_rutas.loc[i, 'cluster'] if 'cluster' in self.df_rutas.columns else np.nan
                        
                        asignaciones.append({
                            'ruta_id': i,
                            'tiempo_predicho': self.tiempos[i],
                            'cluster': cluster_val,
                            'operario': j,
                            'tiempo_efectivo': tiempos_efectivos[i, j]
                        })
            
            self.asignacion = pd.DataFrame(asignaciones)
            
            # Cargas por operario
            self.cargas = self.asignacion.groupby('operario')['tiempo_efectivo'].sum().tolist()
            self.resultado['cargas'] = self.cargas
            
            print(f"Optimización exitosa")
            print(f"   Makespan: {makespan_optimo:.1f} min ({makespan_optimo/60:.1f} horas)")
            print(f"   Cargas: {[f'{c:.1f}' for c in self.cargas]} min")
            
            return True
        else:
            self.resultado = {'status': f'No Óptimo ({estado})'}
            print(f"No se encontró solución óptima: {estado}")
            return False
    
    def comparar_con_actual(self):
        """Compara asignación optimizada vs aleatoria"""
        if self.asignacion is None:
            self.optimizar()
        
        # Simular asignación caótica
        np.random.seed(42)
        
        cargas_caoticas = []
        for j in range(self.n_operarios):
            # Asignar aleatoriamente y calcular carga efectiva
            rutas_asignadas = np.random.choice(self.n_rutas, 
                                              size=self.n_rutas // self.n_operarios + 1, 
                                              replace=False)
            carga = sum(self.tiempos[i] / self.eficiencias[j] 
                       for i in rutas_asignadas if i < self.n_rutas)
            cargas_caoticas.append(carga)
        
        # Ajustar para que todas las rutas estén asignadas
        makespan_caotico = max(cargas_caoticas) * 1.2  # Factor de ineficiencia
        
        horas_extra_opt = max(0, self.resultado['makespan'] - 480) / 60
        horas_extra_caot = max(0, makespan_caotico - 480) / 60
        
        comparacion = {
            'metricas': {
                'makespan_optimo': self.resultado['makespan'],
                'makespan_caotico': makespan_caotico,
                'carga_promedio_optimo': np.mean(self.cargas),
                'carga_promedio_caotico': np.mean(cargas_caoticas),
                'std_carga_optimo': np.std(self.cargas),
                'std_carga_caotico': np.std(cargas_caoticas),
                'horas_extra_optimo': horas_extra_opt,
                'horas_extra_caotico': horas_extra_caot,
                'equidad_optimo': np.std(self.cargas) / np.mean(self.cargas) if np.mean(self.cargas) > 0 else 0,
                'equidad_caotico': np.std(cargas_caoticas) / np.mean(cargas_caoticas) if np.mean(cargas_caoticas) > 0 else 0
            },
            'cargas_optimo': self.cargas,
            'cargas_caotico': cargas_caoticas
        }
        
        comparacion['metricas']['reduccion_horas_extra'] = horas_extra_caot - horas_extra_opt
        comparacion['metricas']['mejora_equidad'] = (
            (comparacion['metricas']['equidad_caotico'] - comparacion['metricas']['equidad_optimo']) / 
            comparacion['metricas']['equidad_caotico'] * 100
            if comparacion['metricas']['equidad_caotico'] > 0 else 0
        )
        
        return comparacion
    
    def visualizar_asignacion(self):
        """Visualiza la asignación optimizada"""
        if self.asignacion is None:
            self.optimizar()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Gráfico 1: Carga por operario
        ax1 = axes[0]
        operarios = [f'Op {j+1}\n(Eff: {self.eficiencias[j]:.0%})' 
                    for j in range(self.n_operarios)]
        colors = ['#2E86AB', '#A23B72', '#F18F01'][:self.n_operarios]
        bars = ax1.bar(operarios, self.cargas, color=colors, edgecolor='black', linewidth=1.2)
        
        ax1.axhline(y=480, color='red', linestyle='--', linewidth=2, 
                   label='Jornada ideal (480 min)')
        ax1.axhline(y=self.resultado['makespan'], color='green', linestyle='--', 
                   linewidth=2, label=f'Makespan: {self.resultado["makespan"]:.1f} min')
        
        ax1.set_ylabel('Tiempo (minutos)', fontsize=12)
        ax1.set_title('Carga Asignada por Operario', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        for bar, carga in zip(bars, self.cargas):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
                    f'{carga:.0f} min\n({carga/60:.1f} h)',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        # Gráfico 2: Distribución de tiempos
        ax2 = axes[1]
        for j in range(self.n_operarios):
            tiempos_op = self.asignacion[self.asignacion['operario'] == j]['tiempo_efectivo']
            ax2.hist(tiempos_op, alpha=0.6, bins=20, color=colors[j],
                    label=f'Op {j+1} (Eff: {self.eficiencias[j]:.0%})',
                    edgecolor='black')
        
        ax2.set_xlabel('Tiempo por ruta (minutos)', fontsize=12)
        ax2.set_ylabel('Frecuencia', fontsize=12)
        ax2.set_title('Distribución de Tiempos por Operario', fontsize=14, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle('Optimización de Asignación de Rutas', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

print("AsignacionJustaOptimizer ADAPTATIVO definido correctamente")

## 5. Ejecución Principal

Flujo completo de la Fase 3:

1. Carga de datos y modelos
2. Predicción de tiempos con Random Forest
3. Optimización de asignación con PuLP
4. Visualización de resultados
5. Simulación de escenarios con SimPy
6. Análisis y recomendaciones

In [ ]:
def ejecutar_fase3(fecha=None, usar_pulp=False, n_simulaciones=300):
    """
    Ejecuta el pipeline completo de la Fase 3 para un día específico.
    
    Args:
        fecha: Fecha a simular (None = día aleatorio)
        usar_pulp: Usar optimización exacta (True) o greedy (False)
        n_simulaciones: Número de iteraciones Monte Carlo
    
    Returns:
        dict: Resultados completos de la jornada
    """
    
    print("=" * 70)
    print("FASE 3: SISTEMA DE ASIGNACIÓN DIARIA DE DESPACHOS")
    print("=" * 70)
    
    # ============================================================
    # PASO 1: Cargar modelos y datos
    # ============================================================
    print("\n[1/5] CARGANDO MODELOS...")
    print("-" * 50)
    
    loader = DataLoaderExacto()
    df_completo, model, scaler, metadata = loader.load_data()
    
    # ============================================================
    # PASO 2: Obtener rutas del día
    # ============================================================
    print("\n[2/5] OBTENIENDO RUTAS DEL DÍA...")
    print("-" * 50)
    
    fecha, rutas_hoy = loader.obtener_rutas_dia(fecha)
    info_dia = loader.obtener_info_dia(fecha)
    
    print(f"   Fecha: {info_dia['fecha']} ({info_dia['dia_semana']})")
    print(f"   Rutas: {info_dia['n_rutas']}")
    print(f"   Productos totales: {info_dia['productos_totales']}")
    print(f"   Valor total: ${info_dia['valor_total']:,.0f}")
    
    # ============================================================
    # PASO 3: Asignar rutas
    # ============================================================
    print(f"\n[3/5] ASIGNANDO RUTAS...")
    print("-" * 50)
    
    asignador = AsignadorDiario(loader)
    asignacion, cargas, makespan, tiempos_base = asignador.asignar_jornada(
        rutas_hoy, usar_pulp=usar_pulp
    )
    
    # Mostrar y visualizar
    asignador.mostrar_resultado(asignacion, cargas, makespan)
    asignador.visualizar_asignacion(asignacion, cargas, makespan, tiempos_base)
    
    # ============================================================
    # PASO 4: Simular escenarios
    # ============================================================
    print(f"\n[4/5] SIMULANDO ESCENARIOS...")
    print("-" * 50)
    
    simulador = SimuladorDespachos(tiempos_base)
    df_simulacion = simulador.simular(n_iter=n_simulaciones)
    simulador.visualizar(df_simulacion)
    
    # ============================================================
    # PASO 5: KPIs y recomendaciones
    # ============================================================
    print(f"\n[5/5] KPIs Y RECOMENDACIONES")
    print("=" * 70)
    
    # Calcular KPIs
    kpis = {
        'fecha': fecha,
        'n_rutas': len(rutas_hoy),
        'makespan_min': makespan,
        'makespan_horas': makespan / 60,
        'horas_extra': max(0, makespan - 480) / 60,
        'dentro_jornada': makespan <= 480,
        'balance_cv': np.std(cargas) / np.mean(cargas) * 100,
        'rutas_luis': len(asignacion['Luis (TÚ)']),
        'carga_luis': cargas[0],
        'carga_op2': cargas[1],
        'carga_op3': cargas[2],
    }
    
    # Mostrar KPIs
    print(f"\nINDICADORES CLAVE:")
    print(f"Rutas totales: {kpis['n_rutas']}")
    print(f"Makespan: {kpis['makespan_min']:.0f} min ({kpis['makespan_horas']:.1f} h)")
    print(f"Horas extra: {kpis['horas_extra']:.1f} h")
    print(f"Balance (CV): {kpis['balance_cv']:.1f}%")
    
    print(f"\nDISTRIBUCIÓN:")
    for j, nombre in enumerate(asignador.nombres):
        print(f"   {nombre}: {len(asignacion[nombre])} rutas | {cargas[j]/60:.1f} horas")
    
    # Simulación KPIs
    print(f"\nROBUSTEZ (simulación):")
    for esc in ['base', 'pico', 'ausentismo']:
        df_esc = df_simulacion[df_simulacion['escenario'] == esc]
        prob_extras = (df_esc['horas_extra'] > 0).sum() / len(df_esc) * 100
        print(f"   {esc.upper()}: {df_esc['makespan'].mean():.0f} min prom | "
              f"{prob_extras:.0f}% prob. horas extra")
    
    # Recomendaciones
    print(f"\nRECOMENDACIONES:")
    
    if makespan > 480:
        print(f"Se necesitan {(makespan-480)/60:.1f} h extras. Considerar:")
        print(f"       - Redistribuir rutas largas")
        print(f"       - Activar apoyo temporal")
    else:
        print(f"Jornada viable en {makespan/60:.1f} horas")
        print(f"Holgura de {(480-makespan)/60:.1f} horas disponible")
    
    if kpis['balance_cv'] > 15:
        print(f"Balance mejorable (CV={kpis['balance_cv']:.0f}%)")
    else:
        print(f"Buen balance de carga (CV={kpis['balance_cv']:.0f}%)")
    
    print(f"\n{'='*70}")
    print(f"FASE 3 COMPLETADA - {fecha}")
    print(f"{'='*70}")
    
    # Retornar resultados
    return {
        'fecha': fecha,
        'info_dia': info_dia,
        'asignacion': asignacion,
        'cargas': cargas,
        'makespan': makespan,
        'tiempos_base': tiempos_base,
        'kpis': kpis,
        'simulacion': df_simulacion,
        'rutas_hoy': rutas_hoy
    }


## 6. Ejecución Principal

Se ejecuta la Fase 3:

- Carga datos y modelos
- Optimiza asignación de rutas
- Simula escenarios operativos
- Genera recomendaciones

In [ ]:
if __name__ == "__main__":
    print("Iniciando Fase 3...")
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Ejecutar para un día (cambia la fecha o déjalo None para día aleatorio)
    resultados = ejecutar_fase3(
        fecha=None,           # None = día aleatorio, o '2026-05-26'
        usar_pulp=False,      # False = Greedy rápido, True = optimización exacta
        n_simulaciones=300    # Iteraciones Monte Carlo
    )
    
    print(f"\nFinalizado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 7. Análisis y Exportación de Resultados

- Explorar en detalle la asignación optimizada
- Exportar resultados a CSV/Excel
- Generar reportes personalizados
- Realizar análisis de sensibilidad

In [ ]:
# Exportar asignación a CSV
os.makedirs('../results', exist_ok=True)

# Crear DataFrame con la asignación detallada
filas = []
for nombre, rutas in resultados['asignacion'].items():
    for ruta in rutas:
        filas.append({
            'fecha': resultados['fecha'],
            'operario': nombre,
            'ruta_id': ruta['ruta_id'],
            'tiempo_asignado_min': ruta['tiempo_asignado'],
            'tiempo_base_min': ruta['tiempo_base'],
            'productos': ruta['productos'],
            'valor': ruta['valor']
        })

df_asignacion = pd.DataFrame(filas)
ruta_export = f"../results/asignacion_{resultados['fecha']}.csv"
df_asignacion.to_csv(ruta_export, index=False)
print(f"Asignación exportada: {ruta_export}")

# Resumen rápido para el equipo
print(f"\nRESUMEN PARA EL EQUIPO - {resultados['fecha']}")
print("=" * 50)
for nombre in ['Luis (TÚ)', 'Operario 2', 'Operario 3']:
    n = len(resultados['asignacion'][nombre])
    carga = resultados['cargas'][list(resultados['asignacion'].keys()).index(nombre)]
    print(f"{nombre}: {n} rutas | {carga:.0f} min ({carga/60:.1f} h)")
print(f"Cierre: {resultados['makespan']:.0f} min ({resultados['makespan']/60:.1f} h)")

## 8. Conclusciones y Recomendaciones

La Fase 3 ha demostrado que la optimización matemática combinada con simulación de escenarios permite:

### Logros Alcanzados
**Reducción de horas extra**: Optimización de asignación reduce significativamente el tiempo extra requerido  
**Equidad laboral**: Distribución balanceada de carga entre operarios con diferentes eficiencias  
**Robustez operativa**: Identificación de puntos críticos bajo escenarios de estrés  
**Priorización estratégica**: Rutas de alto valor (Estrella) asignadas al recurso más eficiente  

### Metodología Aplicada

1. **Predicción:** Random Forest para estimar tiempos de preparación
2. **Optimización:** Programación Lineal Entera (PuLP) para asignación óptima
3. **Simulación:** Monte Carlo (SimPy) para evaluación de escenarios
